In [ ]:
# If needed

# import os
# os.chdir(".../atmosphere-profile-retrieval-dense-nn")
# os.getcwd()

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from paths import LOG_DIR
from pathlib import Path

def get_lists():
    res = {}
    
    for epath in LOG_DIR.rglob("events.*"):
        model_name = epath.parent.name

        ea = EventAccumulator(str(epath))

        ea.Reload()

        tags = ea.Tags()
        if "Eval loss" not in tags["scalars"]:
            continue
        
        scalars = ea.Scalars("Eval loss")

        values = [x.value for x in scalars]
        
        res[model_name] = values

    return res

In [ ]:
lists = get_lists()
lists.keys()

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt


def plot_losses(lists, keys=None):
    if keys is None:
        keys = list(lists.keys())

    min_losses = []
    fig, ax = plt.subplots(layout="constrained", figsize=(6, 4))
    for key in keys:
        losses = lists[key]
        epochs = np.arange(1, 1 + len(losses))
        
        line, = ax.plot(epochs, losses, label=key)

        
        min_loss = np.min(losses)
        ax.axhline(min_loss, linestyle="--", color=line.get_color())

        min_losses.append((min_loss, key))

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Eval loss")
    ax.set_title("Eval loss plot")
    ax.legend()

    plt.show()

    
    losses_sorted = sorted(min_losses)
    for mloss, mname in losses_sorted:
        print(f"{mname}: {mloss:.4f}")
    

    
    fig, ax = plt.subplots(layout="constrained", figsize=(6, 4))

    x = np.arange(1, 1+len(losses_sorted))
    only_losses = [l for l, _ in losses_sorted]

    ax.plot(x, only_losses, color="black")

    for i in range(len(losses_sorted)):
        ax.scatter(x[i], losses_sorted[i][0], label=losses_sorted[i][1])

    ax.legend()

    ax.set_xlabel("N")
    ax.set_ylabel("Min eval loss")
    ax.set_title("Min eval loss plot")

    plt.show()

In [ ]:
# plot_losses(lists, ["1.dense_minimal_inputs - expanding - 2048", "2.dense_full_inputs - expanding - 2048"])
plot_losses(lists, ['BT.expanding_11', 'BT.expanding_8', 'BT.expanding_9', 'BT.expanding_12', 'BT.expanding_10'])

In [ ]:
plot_losses(lists, ['BT-CZ.expanding_10', 'BT-DATE.expanding_10', 'BT-GH.expanding_10', 'BT-GAH.expanding_10', 'BT.expanding_10'])

In [ ]:
plot_losses(lists, ['BT.expanding_11', 'BT-CZ-DATE-GH-GAH-MM.expanding_11'])